In [0]:
# Load the data and set up
import os

from pyspark.sql import functions as F
from pyspark.ml import PipelineModel
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.functions import vector_to_array
from pyspark.sql.window import Window

train = spark.table("workspace.mortgage_risk.model_train")
validation = spark.table("workspace.mortgage_risk.model_validation")

lr_scores = spark.table(
    "workspace.mortgage_risk.validation_scores_lr"
)

model_folder = "/Volumes/workspace/mortgage_risk/model_artifacts"

baseline_model = PipelineModel.load(
    model_folder + "/logistic_regression_v1"
)

gbt_path = model_folder + "/gradient_boosted_trees_v1"
gbt_score_table = "workspace.mortgage_risk.validation_scores_gbt_v1"

In [0]:
# Prepare the training features
preprocessing = PipelineModel(
    stages=baseline_model.stages[:-1]
)

train_ready = preprocessing.transform(train).select(
    "features",
    "label_12m"
)

In [0]:
# Starting choices
#Setting	        Starting value	                     Purpose
#maxIter	            50	                      Number of boosting iterations
#maxDepth	            4	                      Keep individual trees shallow
#stepSize	            0.1	                      Control each tree’s contribution
#maxBins	            64	                      Number of bins available for splits
#minInstancesPerNode	100	                      Avoid splits that create very small groups
#subsamplingRate	    0.8	                      Sample training rows for each tree

In [0]:
gbt = GBTClassifier(
    featuresCol="features",
    labelCol="label_12m",
    maxIter=50,
    maxDepth=4,
    stepSize=0.1,
    maxBins=64,
    minInstancesPerNode=100,
    subsamplingRate=0.8,
    seed=42
)

if os.path.exists(gbt_path):
    gbt_pipeline = PipelineModel.load(gbt_path)
    print("Loaded the saved boosted trees model.")
else:
    gbt_model = gbt.fit(train_ready)

    gbt_pipeline = PipelineModel(
        stages=preprocessing.stages + [gbt_model]
    )

    gbt_pipeline.write().save(gbt_path)
    print("Boosted trees model trained and saved.")

print("Number of trees:", gbt_pipeline.stages[-1].getNumTrees)

Boosted trees model trained and saved.
Number of trees: 50


In [0]:
# Predict on validation data
if spark.catalog.tableExists(gbt_score_table):
    gbt_scores = spark.table(gbt_score_table)
    print("Loaded the saved validation scores.")
else:
    predictions = gbt_pipeline.transform(validation)

    gbt_scores = predictions.withColumn(
        "risk_score",
        vector_to_array("probability")[1]
    ).select(
        "loan_id",
        "prediction_month",
        "label_12m",
        "delinquency_months",
        "risk_score"
    )

    gbt_scores.write.format("delta").mode("error").saveAsTable(
        gbt_score_table
    )

    gbt_scores = spark.table(gbt_score_table)
    print("Validation scores saved.")

Validation scores saved.


In [0]:
# Compare with logistic regression
pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label_12m",
    rawPredictionCol="risk_score",
    metricName="areaUnderPR",
    numBins=10000
)

def check_model(scores, model_name):
    pr_auc = pr_evaluator.evaluate(scores)

    results = scores.agg(
        F.count("*").alias("rows"),
        F.avg("label_12m").alias("event_rate"),
        F.avg("risk_score").alias("average_risk"),
        F.avg(
            F.pow(F.col("risk_score") - F.col("label_12m"), 2)
        ).alias("brier_score")
    )

    results = results.withColumn("model", F.lit(model_name))
    results = results.withColumn("pr_auc", F.lit(pr_auc))

    return results.select(
        "model",
        "rows",
        "pr_auc",
        "brier_score",
        "average_risk",
        "event_rate"
    )

lr_metrics = check_model(lr_scores, "Logistic regression")
gbt_metrics = check_model(gbt_scores, "Gradient boosted trees")

model_comparison = lr_metrics.unionByName(gbt_metrics)

display(model_comparison)

model,rows,pr_auc,brier_score,average_risk,event_rate
Logistic regression,1247263,0.14747397306459945,0.004976100408080951,0.018023464847820036,0.004917968383572671
Gradient boosted trees,1247263,0.17035239191121734,0.005231637010332233,0.03149133744998631,0.004917968383572671


In [0]:
####Compare the models at the same monthly alert budget###

In [0]:
# Load the saved predictions
lr_scores = spark.table("workspace.mortgage_risk.validation_scores_lr")
gbt_scores = spark.table("workspace.mortgage_risk.validation_scores_gbt_v1")


In [0]:
# Add the model names
lr_scores = lr_scores.withColumn("model",F.lit("Logistic regression"))
gbt_scores = gbt_scores.withColumn("model",F.lit("Gradient boosted trees"))

In [0]:
# Prepare the simple delinquency rule
rule_scores = lr_scores.withColumn(
    "model",
    F.lit("Current delinquency rule")
)

rule_scores = rule_scores.withColumn(
    "risk_score",
    F.col("delinquency_months").cast("double")
)

In [0]:
# Put the three approaches together
scores = lr_scores.unionByName(gbt_scores)
scores = scores.unionByName(rule_scores)
scores = scores.withColumn(
    "population",
    F.lit("All eligible loans")
)

In [0]:
# Add a separate comparison for currently healthy loans
healthy_scores = scores.filter(
    (F.col("delinquency_months") == 0) &
    (F.col("model") != "Current delinquency rule")
)

healthy_scores = healthy_scores.withColumn(
    "population",
    F.lit("Current status 00")
)

scores = scores.unionByName(healthy_scores)

In [0]:
# Rank the observations within each month
month_window = Window.partitionBy(
    "model",
    "population",
    "prediction_month"
)

risk_window = month_window.orderBy(
    F.col("risk_score").desc(),
    F.xxhash64("loan_id", "prediction_month").asc(),
    F.col("loan_id").asc()
)

ranked = scores.withColumn(
    "monthly_rows",
    F.count("*").over(month_window)
)

ranked = ranked.withColumn(
    "risk_rank",
    F.row_number().over(risk_window)
)

In [0]:
# Flag the highest-risk 5%
ranked = ranked.withColumn(
    "alert_limit",
    F.ceil(F.col("monthly_rows") * 0.05)
)

ranked = ranked.withColumn(
    "alert",
    F.when(F.col("risk_rank") <= F.col("alert_limit"), 1)
     .otherwise(0)
)

In [0]:
# Count how many positive outcomes the alerts catch
ranked = ranked.withColumn(
    "event_caught",
    F.when(F.col("alert") == 1, F.col("label_12m"))
     .otherwise(0)
)

alert_comparison = ranked.groupBy(
    "model",
    "population"
).agg(
    F.count("*").alias("rows"),
    F.sum("label_12m").alias("events"),
    F.sum("alert").alias("alerts"),
    F.sum("event_caught").alias("events_caught")
)

In [0]:
# Calculate the performance measures
alert_comparison = alert_comparison.withColumn(
    "precision",
    F.when(
        F.col("alerts") > 0,
        F.col("events_caught") / F.col("alerts")
    )
)

alert_comparison = alert_comparison.withColumn(
    "recall",
    F.when(
        F.col("events") > 0,
        F.col("events_caught") / F.col("events")
    )
)

alert_comparison = alert_comparison.withColumn(
    "lift",
    F.when(
        F.col("events") > 0,
        F.col("precision") / (F.col("events") / F.col("rows"))
    )
)

In [0]:
display(
    alert_comparison.orderBy("population", "model")
)

model,population,rows,events,alerts,events_caught,precision,recall,lift
Current delinquency rule,All eligible loans,1247263,6134,62368,2049,0.03285338635197537,0.334039778284969,6.680276038722506
Gradient boosted trees,All eligible loans,1247263,6134,62368,2864,0.04592098512057465,0.4669057711118357,9.337389250805884
Logistic regression,All eligible loans,1247263,6134,62368,2622,0.042040790148794256,0.4274535376589501,8.548405941205669
Gradient boosted trees,Current status 00,1239174,4250,61963,1053,0.016994012555880122,0.24776470588235294,4.954950238804751
Logistic regression,Current status 00,1239174,4250,61963,822,0.013265981311427788,0.19341176470588237,3.867966853084051


In [0]:
table_name = "workspace.mortgage_risk.alert_comparison_gbt_v1"

if spark.catalog.tableExists(table_name):
    print("The alert comparison is already saved.")
else:
    alert_comparison.write.format("delta").mode("error").saveAsTable(
        table_name
    )
    print("Alert comparison saved.")

Alert comparison saved.


In [0]:
monthly_results = ranked.filter(
    F.col("population") == "Current status 00"
).groupBy(
    "prediction_month",
    "model"
).agg(
    F.sum("event_caught").alias("events_caught")
)

In [0]:
# check whether the healthy-loan improvement holds across months
monthly_comparison = monthly_results.groupBy(
    "prediction_month"
).pivot(
    "model",
    ["Logistic regression", "Gradient boosted trees"]
).agg(
    F.first("events_caught")
)

display(
    monthly_comparison.orderBy("prediction_month")
)

prediction_month,Logistic regression,Gradient boosted trees
2022-01-01,88,105
2022-02-01,84,103
2022-03-01,105,127
2022-04-01,109,133
2022-05-01,99,126
2022-06-01,98,126
2022-07-01,89,117
2022-08-01,77,109
2022-09-01,73,107


In [0]:
##### try 100 trees instead of 50 to see whether additional training helps ######

In [0]:
gbt_v2 = GBTClassifier(
    featuresCol="features",
    labelCol="label_12m",
    maxIter=100,
    maxDepth=4,
    stepSize=0.1,
    maxBins=64,
    minInstancesPerNode=100,
    subsamplingRate=0.8,
    seed=42
)

v2_path = model_folder + "/gradient_boosted_trees_v2"

v2_score_table = (
    "workspace.mortgage_risk.validation_scores_gbt_v2"
)

In [0]:
if os.path.exists(v2_path):
    v2_pipeline = PipelineModel.load(v2_path)
    print("Loaded the saved 100-tree model.")
else:
    v2_model = gbt_v2.fit(train_ready)

    v2_pipeline = PipelineModel(
        stages=preprocessing.stages + [v2_model]
    )

    v2_pipeline.write().save(v2_path)
    print("The 100-tree model is trained and saved.")

print("Number of trees:", v2_pipeline.stages[-1].getNumTrees)

The 100-tree model is trained and saved.
Number of trees: 100


In [0]:
# Predict on validation data
v2_predictions = v2_pipeline.transform(validation)

v2_scores = v2_predictions.withColumn(
    "risk_score",
    vector_to_array("probability")[1]
).select(
    "loan_id",
    "prediction_month",
    "label_12m",
    "delinquency_months",
    "risk_score"
)

In [0]:

if spark.catalog.tableExists(v2_score_table):
    v2_scores = spark.table(v2_score_table)
    print("The validation scores are already saved.")
else:
    v2_scores.write.format("delta").mode("error").saveAsTable(
        v2_score_table
    )

    v2_scores = spark.table(v2_score_table)
    print("Validation scores saved.")

Validation scores saved.


In [0]:
# Compare the two versions
v1_scores = spark.table(
    "workspace.mortgage_risk.validation_scores_gbt_v1"
)

v1_metrics = check_model(v1_scores, "GBT v1: 50 trees")
v2_metrics = check_model(v2_scores, "GBT v2: 100 trees")

tuning_results = v1_metrics.unionByName(v2_metrics)

display(tuning_results)

model,rows,pr_auc,brier_score,average_risk,event_rate
GBT v1: 50 trees,1247263,0.17035239191121734,0.005231637010332233,0.03149133744998631,0.004917968383572671
GBT v2: 100 trees,1247263,0.16896966703147875,0.004846918077249144,0.022205237887192215,0.004917968383572671


In [0]:
v1_scores = spark.table(
    "workspace.mortgage_risk.validation_scores_gbt_v1"
)

v2_scores = spark.table(
    "workspace.mortgage_risk.validation_scores_gbt_v2"
)

v1_scores = v1_scores.withColumn(
    "model",
    F.lit("GBT v1: 50 trees")
)

v2_scores = v2_scores.withColumn(
    "model",
    F.lit("GBT v2: 100 trees")
)

In [0]:
version_scores = v1_scores.unionByName(v2_scores)

version_scores = version_scores.withColumn(
    "population",
    F.lit("All eligible loans")
)

healthy_scores = version_scores.filter(
    F.col("delinquency_months") == 0
).withColumn(
    "population",
    F.lit("Current status 00")
)

version_scores = version_scores.unionByName(healthy_scores)

In [0]:
month_window = Window.partitionBy(
    "model", "population", "prediction_month"
)

risk_window = month_window.orderBy(
    F.col("risk_score").desc(),
    F.xxhash64("loan_id", "prediction_month").asc(),
    F.col("loan_id").asc()
)

version_ranked = version_scores.withColumn(
    "monthly_rows",
    F.count("*").over(month_window)
)

version_ranked = version_ranked.withColumn(
    "risk_rank",
    F.row_number().over(risk_window)
)

In [0]:
version_ranked = version_ranked.withColumn(
    "alert_limit",
    F.ceil(F.col("monthly_rows") * 0.05)
)

version_ranked = version_ranked.withColumn(
    "alert",
    F.when(F.col("risk_rank") <= F.col("alert_limit"), 1)
     .otherwise(0)
)

version_ranked = version_ranked.withColumn(
    "event_caught",
    F.when(F.col("alert") == 1, F.col("label_12m"))
     .otherwise(0)
)

In [0]:
version_alerts = version_ranked.groupBy(
    "model", "population"
).agg(
    F.count("*").alias("rows"),
    F.sum("label_12m").alias("events"),
    F.sum("alert").alias("alerts"),
    F.sum("event_caught").alias("events_caught")
)

In [0]:
version_alerts = version_alerts.withColumn(
    "precision",
    F.when(
        F.col("alerts") > 0,
        F.col("events_caught") / F.col("alerts")
    )
)

version_alerts = version_alerts.withColumn(
    "recall",
    F.when(
        F.col("events") > 0,
        F.col("events_caught") / F.col("events")
    )
)

display(
    version_alerts.orderBy("population", "model")
)

model,population,rows,events,alerts,events_caught,precision,recall
GBT v1: 50 trees,All eligible loans,1247263,6134,62368,2864,0.04592098512057465,0.4669057711118357
GBT v2: 100 trees,All eligible loans,1247263,6134,62368,2864,0.04592098512057465,0.4669057711118357
GBT v1: 50 trees,Current status 00,1239174,4250,61963,1053,0.016994012555880122,0.24776470588235294
GBT v2: 100 trees,Current status 00,1239174,4250,61963,1030,0.016622823297774477,0.24235294117647058


In [0]:
                                    ##### 50 trees is Better than 100 trees #########